In [ ]:
import torch
import torchaudio
import os
from torch.utils.data import Dataset, DataLoader
from transformers import HubertModel
import torch.nn as nn
from sklearn.metrics import accuracy_score
from tqdm import tqdm

# === CONFIG ===
class Config:
    SAMPLE_RATE = 16000
    MODEL_NAME = "facebook/hubert-base-ls960"
    BATCH_SIZE = 4
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    DATA_DIR = "data/trimmed_fan"

config = Config()

# === MODEL LOADING ===
class HubertClassifier(nn.Module):
    def __init__(self, hidden_dim=768, num_classes=2):
        super(HubertClassifier, self).__init__()
        self.hubert = HubertModel.from_pretrained(config.MODEL_NAME)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes)
        )

    def forward(self, input_values):
        with torch.no_grad():
            features = self.hubert(input_values).last_hidden_state
        pooled = torch.mean(features, dim=1)
        logits = self.classifier(pooled)
        return logits

# Load the pretrained HuBERT model
model = HubertClassifier().to(config.DEVICE)
model.load_state_dict(torch.load("hubert_transformer_synthetic.pt"))
model.eval()

# === DATASET FOR RAW AUDIO ===
class AudioDataset(Dataset):
    def __init__(self, root_dir):
        self.samples = []

        for label_str in ["normal", "abnormal"]:
            label = 0 if label_str == "normal" else 1
            class_dir = os.path.join(root_dir, label_str)
            for subdir in os.listdir(class_dir):
                subdir_path = os.path.join(class_dir, subdir)
                for file in os.listdir(subdir_path):
                    if file.endswith(".wav"):
                        self.samples.append((os.path.join(subdir_path, file), label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        file_path, label = self.samples[idx]
        waveform, sample_rate = torchaudio.load(file_path)
        # Resample to the sample rate expected by HuBERT model
        waveform = torchaudio.functional.resample(waveform, sample_rate, config.SAMPLE_RATE)
        return waveform.squeeze(0), label

# === DATA LOADER ===
dataset = AudioDataset(config.DATA_DIR)
dataloader = DataLoader(dataset, batch_size=config.BATCH_SIZE, shuffle=False)

# === EVALUATION ===
def evaluate():
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for waveforms, labels in tqdm(dataloader, desc="Evaluating"):
            waveforms = waveforms.to(config.DEVICE)
            labels = labels.to(config.DEVICE)

            # Run inference on the waveform
            outputs = model(waveforms)
            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # Calculate accuracy
    accuracy = accuracy_score(all_labels, all_preds) * 100
    print(f"Accuracy on real data: {accuracy:.2f}%")

# === MAIN ===
if __name__ == '__main__':
    evaluate()
